# Character and Word Frequency Analysis

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Token frequency counting and vocabulary construction are fundamental stages in Natural Language Processing (NLP) and information retrieval. In this notebook, we implement character-level and word-level frequency counters from first principles, comparing algorithmic paradigms across imperative dictionaries, `.get()` fallbacks, `defaultdict`, and `collections.Counter`.

### Objectives
1. Implement case-insensitive character counting across alphabetic strings.
2. Read text files defensively using context managers and `pathlib.Path`.
3. Normalize text corpora by stripping punctuation, handling contractions, and tokenizing whitespace.
4. Analyze term frequency distributions on real benchmark datasets.


## 1. Character Frequency in Words (Problem 1)

**Specification:** Given an input string containing alphabetic characters $[a-zA-Z]$, return a dictionary mapping each lowercased character to its total frequency count.


In [ ]:
from collections import defaultdict


def count_characters_imperative(word: str) -> dict[str, int]:
    """Count character frequencies using an imperative if-else dictionary check."""
    result: dict[str, int] = {}
    normalized = word.lower()
    for ch in normalized:
        if not ch.isalpha():
            continue
        if ch in result:
            result[ch] += 1
        else:
            result[ch] = 1
    return result


def count_characters_get(word: str) -> dict[str, int]:
    """Count character frequencies using dict.get() default fallback."""
    result: dict[str, int] = {}
    for ch in word.lower():
        if ch.isalpha():
            result[ch] = result.get(ch, 0) + 1
    return result


def count_characters_defaultdict(word: str) -> dict[str, int]:
    """Count character frequencies using collections.defaultdict(int)."""
    result: defaultdict[str, int] = defaultdict(int)
    for ch in word.lower():
        if ch.isalpha():
            result[ch] += 1
    return dict(result)


# Test cases
word_a = "Happiness"
word_b = "Baby"

res_a = count_characters_imperative(word_a)
res_b = count_characters_get(word_b)

print(f"Character counts for '{word_a}':", res_a)
print(f"Character counts for '{word_b}':", res_b)

assert res_a == {"h": 1, "a": 1, "p": 2, "i": 1, "n": 1, "e": 1, "s": 2}
assert res_b == {"b": 2, "a": 1, "y": 1}
assert count_characters_defaultdict(word_a) == res_a

## 2. Word Frequency in Text Files (Problem 2)

**Specification:** Read an external plain text file, sanitize each sentence by lowercasing and stripping punctuation, tokenize into individual words, and return a dictionary mapping each word to its frequency.


In [ ]:
import string
from pathlib import Path


def sanitize_and_tokenize(text: str) -> list[str]:
    """Normalize text by lowercasing and removing standard punctuation marks."""
    translator = str.maketrans(string.punctuation, " " * len(string.punctuation))
    cleaned = text.translate(translator).lower()
    return cleaned.split()


def count_words_in_text(text: str) -> dict[str, int]:
    """Compute word frequency dictionary from raw text string."""
    tokens = sanitize_and_tokenize(text)
    counts: dict[str, int] = {}
    for token in tokens:
        counts[token] = counts.get(token, 0) + 1
    return counts


# Quick verification on sample sentence
sample_sentence = "AI is a topic that is very interesting. However, it is not easy."
sample_counts = count_words_in_text(sample_sentence)
print("Sample sentence counts:", sample_counts)
assert sample_counts["is"] == 3
assert sample_counts["interesting"] == 1
assert sample_counts["however"] == 1

In [ ]:
def count_words_from_file(file_path: Path | str) -> dict[str, int]:
    """Load text file safely via context manager and compute word frequencies."""
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    with path.open(encoding="utf-8") as f:
        content = f.read()

    return count_words_in_text(content)


# Process session dataset
corpus_path = Path("../data/sample_text.txt")
corpus_counts = count_words_from_file(corpus_path)

print(f"Total unique vocabulary size: {len(corpus_counts)}")
top_5_words = sorted(corpus_counts.items(), key=lambda kv: kv[1], reverse=True)[:5]
print("Top 5 most frequent terms:", top_5_words)

assert len(corpus_counts) > 30
assert "python" in corpus_counts
assert "and" in corpus_counts

## 3. Complexity and Statistical Properties

- **Time Complexity:**
  - Tokenization: $O(N)$ where $N$ is the number of characters in the file.
  - Dictionary Insertion: Average $O(1)$ per word via hash table lookups, totaling $O(W)$ where $W$ is total word count.
  - Overall Time Complexity: $O(N + W) = O(N)$.
- **Space Complexity:**
  - Hash map storage requires $O(V)$ auxiliary memory where $V$ is vocabulary size ($V \le W$).
